In [1]:
from implement_LLM import oracle_retriever
from metrics import compute_all_metrics
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Oracle retriever

In [2]:
data_oracle=oracle_retriever('eval_set_v3_clean_sampled.jsonl',all_chunks=None,name_model='llama-3.1-8b-instant')

  0%|          | 0/101 [00:00<?, ?it/s]

In [3]:
print(data_oracle.info())

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   question_id          101 non-null    str    
 1   chunk_id             93 non-null     float64
 2   source_chunk_title   101 non-null    str    
 3   question             101 non-null    str    
 4   ground_truth_answer  101 non-null    str    
 5   notes                23 non-null     str    
 6   relevant_chunk_ids   101 non-null    object 
 7   ground_truth_text    101 non-null    str    
 8   answer_type          101 non-null    str    
 9   reasoning_chain      6 non-null      str    
 10  negative_type        9 non-null      str    
 11  base_question_id     6 non-null      str    
 12  referenced_title     2 non-null      str    
 13  false_assumption     2 non-null      str    
 14  expected_behavior    2 non-null      str    
 15  llm_answer           101 non-null    str    
 16  g

In [4]:
data_oracle.to_csv('oracle_retrieved_eval_set_with_answers.csv')

In [6]:
data_oracle=pd.read_csv('oracle_retrieved_eval_set_with_answers.csv')

In [10]:
import json
import ast

def parse_chunks(val):
    try:
        return json.loads(val)
    except:
        return ast.literal_eval(val)

data_oracle['relevant_chunks'] = data_oracle['relevant_chunks'].apply(parse_chunks)

In [ ]:

result=compute_all_metrics(is_generation_metrics=True,
                    data_samples=data_oracle,
                    model_name='llama-3.1-8b-instant',
                    queries_column='question',
                    answers_column='llm_answer',
                    chunks_column='relevant_chunks',
                    ground_truth_column='ground_truth_text',
                    is_time_metrics=True,
                    list_columns_time=['generation_time','e2e_latency']
                    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/303 [00:00<?, ?it/s]

Exception raised in Job[6]: OutputParserException(Invalid json output: Here's the analysis of the complexity of each sentence in the answer and breaking down each sentence into one or more fully understandable statements.

Input:
{
  "question": "Who handed Gon his Hunter License after he passed the exam?",
  "answer": "Satotz"
}

Output:
{
  "statements": [
    "Satotz handed Gon his Hunter License.",
    "Gon passed the exam."
  ]
}

Explanation:
- The answer "Satotz" is a single word, so it's not possible to break it down into more statements.
- However, we can infer that Satotz handed Gon his Hunter License after Gon passed the exam, so we can create two separate statements.

Note: The output is in JSON format, complying with the specified schema.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[7]: OutputParserException(Invalid json output: Here's the analysis of the complexity of each sentence in t

In [17]:
print(result['time'])

{'generation_time': {'mean': np.float64(5.523917951484529), 'median': np.float64(6.2707615999970585), 'p95': np.float64(9.529616400017405)}, 'e2e_latency': {'mean': np.float64(5.5239249930652505), 'median': np.float64(6.270770000002813), 'p95': np.float64(9.529621399997268)}}


In [11]:
generation_df = result['generation'][['faithfulness', 'answer_correctness', 'answer_relevancy']]
data_oracle_with_metrics = pd.concat([data_oracle.reset_index(drop=True), generation_df.reset_index(drop=True)], axis=1)

In [18]:
data_oracle_with_metrics.to_csv('data_oracle_with_metrics.csv')

# Retriever Eval

## BGE-M3